In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib as mpl
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.layer_extension import DataFilterExtension
from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap
from functools import reduce
from constants import PATH
from utils import load_admin_data
from manywidgets import Column, Row
from manywidgets.lonboard import LayerToggle, LayerFilter


In [ ]:
iso_code = "MOZ"


## Add data from pre-processing steps

In [ ]:
iso_lower = iso_code.lower()
exposure_base = f"{PATH}{iso_lower}/236_exposure/{iso_lower}_exposure_tab_mapaction_"

pop_exposure = pd.read_excel(f"{exposure_base}population-admin2.xlsx")
health_facilities_exposure = pd.read_excel(f"{exposure_base}health-facilities.xlsx")
health_facilities_flood_exposure = pd.read_excel(f"{exposure_base}health-facilities-admin2-flood-current.xlsx")


In [3]:
admin2_dfs = [pop_exposure, health_facilities_flood_exposure]
admin2_dfs = [admin2_dfs[0]] + [d.drop(columns=["ISO", "ADM1_PCODE"], errors="ignore") for d in admin2_dfs[1:]]

df = reduce(lambda left, right: pd.merge(left, right, on="ADM2_PCODE", how="outer"), admin2_dfs)

In [ ]:
admin2_boundaries = load_admin_data(use_gadm_boundaries=True, iso_code=iso_code, admin_level="admin2")
gdf = admin2_boundaries.merge(df, on="ADM2_PCODE", how="left")


## Health facility points

In [ ]:
health_sites = gpd.read_file(f"{PATH}{iso_lower}/215_heal/{iso_lower}_heal_pt_healthsites.geojson")
health_sites = health_sites.drop_duplicates(subset="osm_id")

health_gdf = health_sites.merge(health_facilities_exposure, on="osm_id", how="inner")


In [6]:
FLOOD_COLUMN = "flood_current"
CMAP = mpl.colormaps["YlOrRd"]


def colors_for_hazard(column):
    values = gdf[column].fillna(0).to_numpy(dtype=float)
    value_range = values.max() - values.min()
    normalized = (values - values.min()) / value_range if value_range else np.zeros_like(values)
    return apply_continuous_cmap(normalized, CMAP)


layer_flood = PolygonLayer.from_geopandas(
    gdf,
    get_line_width=20,
    line_width_min_pixels=0.2,
    get_fill_color=colors_for_hazard(FLOOD_COLUMN),
    visible=True,
)
toggle_flood = LayerToggle(layer=layer_flood, value=True, label="Flood: avg. annual % of population exposed")

FLOOD_RP_ORDER = ["10yr", "50yr", "100yr", "500yr", "1000yr", "no_exposure"]
present_categories = set(health_gdf["flood_current"].unique())
ordered_categories = [c for c in FLOOD_RP_ORDER if c in present_categories]

flood_categories = pd.Categorical(health_gdf["flood_current"], categories=ordered_categories)
flood_codes = flood_categories.codes.astype("int32")
category_pairs = [[i, label] for i, label in enumerate(flood_categories.categories)]

health_layer = ScatterplotLayer.from_geopandas(
    health_gdf,
    get_radius=800,
    radius_min_pixels=2,
    get_fill_color=apply_categorical_cmap(
        health_gdf["flood_current"],
        {
            "no_exposure": [180, 180, 180],
            "10yr": [227, 26, 28],
            "50yr": [252, 78, 42],
            "100yr": [253, 141, 60],
            "500yr": [254, 217, 118],
            "1000yr": [255, 237, 160],
        },
    ),
    extensions=[DataFilterExtension(category_size=1)],
    get_filter_category=flood_codes,
    filter_categories=list(range(len(category_pairs))),
    visible=True,
)
toggle_health = LayerToggle(layer=health_layer, value=True, label="Health facilities")

no_exposure_index = ordered_categories.index("no_exposure") if "no_exposure" in ordered_categories else None
default_checked = [i for i in range(len(category_pairs)) if i != no_exposure_index]

health_filter = LayerFilter(
    health_layer,
    categories=category_pairs,
    value=default_checked,
    label="Facility flood exposure (most frequent return period affecting it)",
)

m = Map([layer_flood, health_layer])
Column(
    Row(toggle_flood, toggle_health, gap="24px"),
    health_filter,
    m,
)